# Cluster profile visualisation (fig 2.2)

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path(os.getcwd()).resolve().parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from src.preprocess import preprocess_pipeline

FIGURES_DIR = PROJECT / 'thesis' / 'figures'
FIGURES_DIR.mkdir(exist_ok=True, parents=True)

FEATURE_COLS = [
    'energy', 'danceability', 'valence',
    'acousticness', 'instrumentalness', 'speechiness',
    'loudness', 'tempo',
]
FEATURE_LABELS = {
    'energy': 'Energy',
    'danceability': 'Danceability',
    'valence': 'Valence\n(positivity)',
    'acousticness': 'Acousticness',
    'instrumentalness': 'Instrumental',
    'speechiness': 'Speechiness',
    'loudness': 'Loudness\n(dB)',
    'tempo': 'Tempo\n(BPM)',
}
print('setup done.')

## 2. Load catalogue and cluster

In [ ]:
df, audio_features, scaler, kmeans = preprocess_pipeline('data/spotify_data.csv')
print(f'rows: {len(df):,}    clusters: {kmeans.n_clusters}')

## 3. Per-cluster feature means + top genres

In [ ]:
profile = df.groupby('cluster')[FEATURE_COLS].mean()
sizes = df['cluster'].value_counts().sort_index()
profile['size'] = sizes

top_genres = df.groupby('cluster')['genre'].agg(
    lambda s: ', '.join(s.value_counts().head(2).index.tolist())
)
profile['top_genres'] = top_genres

profile = profile.sort_values('energy', ascending=False)
profile.head()

## 4. Build heatmap

In [ ]:
display = profile[FEATURE_COLS].copy()
norm = (display - display.min()) / (display.max() - display.min())

fig, ax = plt.subplots(figsize=(11.5, 8.5))
cmap = LinearSegmentedColormap.from_list(
    'thesis', ['#f7f7f7', '#fee08b', '#fc8d59', '#a50f15']
)
im = ax.imshow(norm.values, aspect='auto', cmap=cmap, vmin=0, vmax=1)

for i in range(len(profile)):
    for j, col in enumerate(FEATURE_COLS):
        val = profile.iloc[i][col]
        if col == 'tempo':
            txt = f'{val:.0f}'
        elif col == 'loudness':
            txt = f'{val:.1f}'
        else:
            txt = f'{val:.2f}'
        colour = 'white' if norm.iloc[i, j] > 0.6 else '#222'
        ax.text(j, i, txt, ha='center', va='center', fontsize=8.5, color=colour)

yticklabels = [
    f'Cluster {int(c):>2}  ({int(profile.loc[c, "size"]):>5,d} songs) - {profile.loc[c, "top_genres"]}'
    for c in profile.index
]
ax.set_yticks(range(len(profile)))
ax.set_yticklabels(yticklabels, fontsize=9, fontfamily='monospace')

ax.set_xticks(range(len(FEATURE_COLS)))
ax.set_xticklabels([FEATURE_LABELS[c] for c in FEATURE_COLS], fontsize=10)

ax.set_title(
    f'K-Means cluster profiles (k = {kmeans.n_clusters})\n'
    'rows sorted by mean energy, cell values are per-cluster feature means; '
    'cell colour is normalised within each column',
    fontsize=11,
)
ax.tick_params(axis='x', which='both', top=False, bottom=False)
ax.tick_params(axis='y', which='both', left=False)
ax.set_yticks([y + 0.5 for y in range(len(profile) - 1)], minor=True)
ax.grid(axis='y', which='minor', color='white', linewidth=0.6)

plt.tight_layout()
out = FIGURES_DIR / 'fig_2_2.png'
plt.savefig(out, dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print(f'saved: {out}')

## 5. Diagnostics — find apparent-duplicate cluster pairs

In [ ]:
from scipy.spatial.distance import pdist, squareform

centroids = kmeans.cluster_centers_
dist_matrix = squareform(pdist(centroids))
np.fill_diagonal(dist_matrix, np.inf)

print(f'centroid distance — mean: {dist_matrix[np.isfinite(dist_matrix)].mean():.2f}')
print(f'centroid distance — min : {dist_matrix.min():.2f}')
print(f'centroid distance — max : {dist_matrix[np.isfinite(dist_matrix)].max():.2f}')
print()

pairs = []
for i in range(len(centroids)):
    for j in range(i + 1, len(centroids)):
        pairs.append((i, j, dist_matrix[i, j]))
pairs.sort(key=lambda x: x[2])

print('Five closest cluster pairs (smallest centroid distance first):')
for i, j, d in pairs[:5]:
    g_i = profile.loc[i, 'top_genres']
    g_j = profile.loc[j, 'top_genres']
    print(f'  cluster {i:>2} <-> cluster {j:>2}   distance = {d:.2f}')
    print(f'    cluster {i}: {g_i}')
    print(f'    cluster {j}: {g_j}')

## 6. Silhouette score (optional, takes ~30 s)

In [ ]:
from sklearn.metrics import silhouette_score

rng = np.random.default_rng(42)
idx = rng.choice(len(df), size=30_000, replace=False)
X = audio_features[idx]
labels = df.iloc[idx]['cluster'].values
score = silhouette_score(X, labels, sample_size=10_000, random_state=42)
print(f'silhouette score (sample of 30,000 songs): {score:.4f}')